# 6.1 Near-duplicate candidates

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv('../data/processed/task5/train_clean_5_1.csv')
print("Dataset shape:", df.shape)
print("ad_id unique:", df['ad_id'].is_unique)

Dataset shape: (5109, 66)
ad_id unique: True


Key for a duplicate candidate: same `model_key`, `price`, `loc_city` and `storage_gb`. Ads with an unknown model or no city are skipped, because many different phones share `unknown`.

In [2]:
# ads that are allowed to be grouped
eligible = (
    (df['model_key'] != 'unknown')
    & df['model_key'].notna()
    & df['loc_city'].notna()
)

print("Eligible ads:", eligible.sum())
print("Skipped ads:", (~eligible).sum())

Eligible ads: 4641
Skipped ads: 468


In [3]:
# storage_gb is missing for some ads, fill with -1 only inside the key
# so they can still match each other
df['storage_key'] = df['storage_gb'].fillna(-1)
key = ['model_key', 'price', 'loc_city', 'storage_key']

df['dup_group_size'] = 1
df.loc[eligible, 'dup_group_size'] = df[eligible].groupby(key)['ad_id'].transform('size')

is_candidate = eligible & (df['dup_group_size'] > 1)
print("Candidate ads:", is_candidate.sum(), f"({is_candidate.mean() * 100:.1f}%)")

Candidate ads: 575 (11.3%)


In [4]:
# give every group an id: dup_0001, dup_0002, ...
group_codes = df.loc[is_candidate, key].astype(str).agg('|'.join, axis=1)
group_numbers = pd.factorize(group_codes)[0] + 1

df['dup_candidate_id'] = None
df.loc[is_candidate, 'dup_candidate_id'] = ['dup_%04d' % n for n in group_numbers]

print("Candidate groups:", df['dup_candidate_id'].nunique())
print("Extra ads (group size - 1):", is_candidate.sum() - df['dup_candidate_id'].nunique())
print()
print(df.loc[is_candidate].drop_duplicates('dup_candidate_id')['dup_group_size'].value_counts().sort_index())

Candidate groups: 238
Extra ads (group size - 1): 337

dup_group_size
2     183
3      34
4       9
5       6
6       4
7       1
10      1
Name: count, dtype: int64


Most of the `storage_gb` values were imputed, not written in the ad. A group where every ad has a real (observed) storage is a stronger candidate, so we mark that.

In [5]:
observed_storage = (df['storage_gb_was_imputed'] == False) & df['storage_gb'].notna()

all_observed = (
    df[is_candidate]
    .assign(observed=observed_storage[is_candidate])
    .groupby('dup_candidate_id')['observed']
    .transform('all')
)

df['dup_key_strength'] = None
df.loc[is_candidate, 'dup_key_strength'] = np.where(all_observed, 'strong', 'weak')

print(df.drop_duplicates('dup_candidate_id').dropna(subset=['dup_candidate_id'])['dup_key_strength'].value_counts())

dup_key_strength
weak      235
strong      3
Name: count, dtype: int64


In [6]:
groups = (
    df[is_candidate]
    .groupby('dup_candidate_id')
    .agg(
        n_ads=('ad_id', 'size'),
        model_key=('model_key', 'first'),
        price=('price', 'first'),
        loc_city=('loc_city', 'first'),
        storage_gb=('storage_gb', 'first'),
        strength=('dup_key_strength', 'first'),
        n_titles=('title_normalized', 'nunique'),
        n_areas=('loc_area', 'nunique'),
        n_title_groups=('title_group', 'nunique'),
    )
    .sort_values('n_ads', ascending=False)
)

groups.head(10)

,n_ads,model_key,price,loc_city,storage_gb,strength,n_titles,n_areas,n_title_groups
dup_candidate_id,,,,,,,,,
dup_0217,10,apple iphone 13,19500.0,الجيزة,128.0,weak,2,10,2
dup_0017,7,apple iphone 17 pro max,85000.0,القاهرة,NaN,weak,3,5,3
dup_0045,6,apple iphone 11,11000.0,القاهرة,128.0,weak,3,6,3
dup_0055,6,apple iphone 13,23000.0,القاهرة,128.0,weak,2,6,2
dup_0072,6,apple iphone 17 pro max,86000.0,القاهرة,NaN,weak,2,5,2
dup_0009,6,apple iphone 13,25000.0,القاهرة,128.0,weak,3,5,3
dup_0048,5,apple iphone 13,26000.0,القاهرة,128.0,weak,3,5,3
dup_0061,5,apple iphone 13,24000.0,القاهرة,128.0,weak,2,5,2
dup_0013,5,apple iphone 16 pro max,60000.0,القاهرة,256.0,weak,1,5,1


In [7]:
# groups that cross title_group can leak between train and test in grouped CV
print("Groups with more than one title_group:", (groups['n_title_groups'] > 1).sum(), "of", len(groups))
print("Groups with more than one area:", (groups['n_areas'] > 1).sum(), "of", len(groups))

Groups with more than one title_group: 168 of 238
Groups with more than one area: 197 of 238


In [8]:
Path('../data/processed/task6').mkdir(parents=True, exist_ok=True)

df.drop(columns=['storage_key']).to_csv('../data/processed/task6/dup_candidates_6_1.csv', index=False)
groups.to_csv('../data/processed/task6/dup_groups_6_1.csv')

print("Saved candidates and groups")

Saved candidates and groups


## Review sheet for 6.2 (50 groups)

In [9]:
# random 50 groups, same share of strong/weak as in the data
all_groups = groups.reset_index()
strong = all_groups[all_groups['strength'] == 'strong']
weak = all_groups[all_groups['strength'] == 'weak']

n_strong = max(1, round(50 * len(strong) / len(all_groups)))
n_weak = 50 - n_strong

sample = pd.concat([
    strong.sample(n_strong, random_state=42),
    weak.sample(n_weak, random_state=42),
])
sample = sample.sample(frac=1, random_state=42).reset_index(drop=True)
sample['review_no'] = sample.index + 1

print("Groups sampled:", len(sample))
print(sample['strength'].value_counts())

Groups sampled: 50
strength
weak      49
strong     1
Name: count, dtype: int64


In [10]:
review = df[df['dup_candidate_id'].isin(sample['dup_candidate_id'])].merge(
    sample[['dup_candidate_id', 'review_no']], on='dup_candidate_id'
).sort_values(['review_no', 'ad_id'])

cols = ['review_no', 'dup_candidate_id', 'dup_key_strength', 'dup_group_size', 'ad_id', 'title',
        'price', 'loc_city', 'loc_area', 'storage_gb', 'storage_gb_was_imputed', 'condition', 'ad_url']
review_sheet = review[cols].copy()
review_sheet['ad_url'] = 'https://www.dubizzle.com.eg' + review_sheet['ad_url'].astype(str)

# to fill by hand: dup / not_dup / partial (one value per group, in its first row)
review_sheet['group_decision'] = ''
review_sheet['notes'] = ''

review_sheet.to_csv('../data/processed/task6/dup_review_50.csv', index=False, encoding='utf-8-sig')
print("Groups:", review_sheet['review_no'].nunique(), "| rows:", len(review_sheet))
review_sheet.head(8)

Groups: 50 | rows: 118


,review_no,dup_candidate_id,dup_key_strength,dup_group_size,ad_id,title,price,loc_city,loc_area,storage_gb,storage_gb_was_imputed,condition,ad_url,group_decision,notes
106,1,dup_0231,weak,2,208957210,هونر 600,30000.0,الإسكندرية,محرّم بيك,256.0,True,NaN,https://www.dubizzle.com.eg/ad/%D9%87%D9%88%D9...,,
105,1,dup_0231,weak,2,208957216,هونر 600,30000.0,الإسكندرية,عجمي,256.0,True,NaN,https://www.dubizzle.com.eg/ad/%D9%87%D9%88%D9...,,
55,2,dup_0155,weak,2,208986166,آبل - آيفون اكس ار,7500.0,الجيزة,المهندسين,64.0,True,NaN,https://www.dubizzle.com.eg/ad/%D8%A2%D8%A8%D9...,,
43,2,dup_0155,weak,2,208990688,آبل - آيفون اكس ار,7500.0,الجيزة,6 اكتوبر,64.0,True,NaN,https://www.dubizzle.com.eg/ad/%D8%A2%D8%A8%D9...,,
115,3,dup_0180,weak,4,208454592,Apple - iPhone 13,27000.0,القاهرة,المنيل,128.0,True,NaN,https://www.dubizzle.com.eg/ad/apple-iphone-13...,,
57,3,dup_0180,weak,4,208742896,Apple - iPhone 13,27000.0,القاهرة,المعادي,128.0,True,NaN,https://www.dubizzle.com.eg/ad/apple-iphone-13...,,
101,3,dup_0180,weak,4,208958439,آبل - آيفون 13,27000.0,القاهرة,حدائق الزيتون,128.0,True,NaN,https://www.dubizzle.com.eg/ad/%D8%A2%D8%A8%D9...,,
75,3,dup_0180,weak,4,208973095,Apple - iPhone 13,27000.0,القاهرة,مدينة نصر,128.0,True,NaN,https://www.dubizzle.com.eg/ad/apple-iphone-13...,,
